In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv

def find_repo_root(start_path: Path, marker=".git", max_up=5) -> Path:
    """
    Walks up from start_path up to max_up levels to find a folder containing `marker`.
    Returns the Path to the repo root or raises FileNotFoundError.
    """
    current = start_path.resolve()
    for _ in range(max_up):
        if (current / marker).exists():
            return current
        current = current.parent
    raise FileNotFoundError(f"Could not find repo root containing {marker} within {max_up} levels up from {start_path}")

# In your notebook, get current working dir (where notebook runs)
cwd = Path().resolve()

try:
    repo_root = find_repo_root(cwd)
except FileNotFoundError:
    # Fallback: If no .git folder, fallback to known repo folder name
    repo_root = cwd
    while repo_root.name != "league-logger" and repo_root != repo_root.parent:
        repo_root = repo_root.parent
    if repo_root.name != "league-logger":
        raise RuntimeError("Could not find league-logger folder as repo root")

src_path = repo_root / "src"

# Add /src to sys.path if not already added
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

# Load .env
load_dotenv(dotenv_path=repo_root / ".env")

# Now import your modules
import api_handler
import sync

print(f"Repo root found at: {repo_root}")
print(f"/src added to sys.path: {src_path}")


In [ ]:
import os

# Handle dynamic pathing (safe in notebooks and scripts)
try:
    repo_root = Path(__file__).resolve().parents[1]
except NameError:
    repo_root = Path.cwd().parents[1]  # you're likely in /notebooks

# Add /src to sys.path for importing sync
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

# Load .env
env_path = repo_root / ".env"
load_dotenv(dotenv_path=env_path)

RIOT_API_KEY = os.getenv("RIOT_API_KEY")
REGION_ROUTING = os.getenv("REGION_ROUTING")

print("RIOT_API_KEY:", os.getenv("RIOT_API_KEY"))

if not RIOT_API_KEY:
    raise ValueError("RIOT_API_KEY not found in .env")

# Import the sync function now that pathing is handled
from sync import sync_user_data

# === CONFIG ===
SUMMONER_NAME = "RainbowThenga"
BASE_DATA_PATH = repo_root / "data" / "users"

# Optional filters — leave as None if not filtering
CHAMPION_NAME = None
START_TIME = None  # You could use something like int(datetime(2024, 6, 1).timestamp())
END_TIME = None


load_dotenv()
RIOT_API_KEY = os.getenv("RIOT_API_KEY")

if not RIOT_API_KEY:
    raise ValueError("RIOT_API_KEY not found in .env")

HEADERS = {
    "X-Riot-Token": RIOT_API_KEY
}

# Change these as needed
REGION_ROUTING = "europe"   # For match/v5 endpoints (matches & timelines)
PLATFORM_ROUTING = "euw1"      # For summoner-v4 endpoints (summoner info)


# Run the sync
sync_user_data(
    summoner_name=SUMMONER_NAME,
    base_data_path=BASE_DATA_PATH,
    champion_name=CHAMPION_NAME,
    start_time=START_TIME,
    end_time=END_TIME
)

In [ ]:
import os
import requests
from dotenv import load_dotenv
from pathlib import Path

# Load .env (adjust path if needed)
load_dotenv(dotenv_path=Path.cwd().parents[1] / ".env")

RIOT_API_KEY = os.getenv("RIOT_API_KEY")
HEADERS = {"X-Riot-Token": RIOT_API_KEY}

# Use test summoner (Caps) on EUW
summoner_name = "Caps"
platform_routing = "euw1"
url = f"https://{platform_routing}.api.riotgames.com/lol/summoner/v4/summoners/by-name/{summoner_name}"

response = requests.get(url, headers=HEADERS)

print("Status Code:", response.status_code)
print("Response:", response.json())

In [ ]:
print("RIOT_API_KEY:", RIOT_API_KEY[:10] + "..." if RIOT_API_KEY else "None")